# 1. Prueba videos

In [ ]:
# 🔹 Importar las librerías
import cv2
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import folium 

In [ ]:
# 🔹 Función para convertir coordenadas raster a coordenadas georreferenciadas
def raster_to_geo(cx, cy, width, height, transformer, xmin_geo, ymin_geo, xmax_geo, ymax_geo):
    try:
        x_geo = xmin_geo + (cx / width) * (xmax_geo - xmin_geo)
        y_geo = ymin_geo + (1 - cy / height) * (ymax_geo - ymin_geo)
        lon, lat = transformer.transform(x_geo, y_geo)  # Transformación de coordenadas
        return lat, lon
    except Exception as e:
        print(f"Error en transformación de coordenadas: {e}")
        return None, None

In [ ]:
# 🔹 Ruta del video (Cambia esto por la ruta de tu video)
video_path = "./Mesa3D/20241128_150153.mp4"

In [ ]:
# 🔹 Factor de escala para reducir la resolución (Ejemplo: 0.5 para reducir a la mitad)
scale_factor = 0.3

In [ ]:
# 🔹 Definir área de corte
ymin, xmin, ymax, xmax = 125, 50, 500, 300

In [ ]:
# 🔹 Definir georreferenciación de la extensión del video
xmin_geo, ymin_geo, xmax_geo, ymax_geo = 416316.969, 2079317.310, 617400.705, 2256323.915

In [ ]:
# 🔹 Transformador de coordenadas UTM zona 14N a WGS84
transformer = Transformer.from_crs(32614, 4326, always_xy=True)

In [ ]:
# 🔹 Abrir el video con OpenCV
cap = cv2.VideoCapture(video_path)

In [ ]:
# Verificar si el video se abrió correctamente
if not cap.isOpened():
    print("Error: No se pudo abrir el video.")
else:
    print("Reproduciendo video...")

In [ ]:
# 🔹 Definir rango de color para detectar plastilina amarilla y verde (en HSV)
lower_yellow = np.array([20, 100, 100])
upper_yellow = np.array([30, 255, 255])

lower_green = np.array([40, 40, 40])
upper_green = np.array([80, 255, 255])

In [ ]:
# 🔹 Leer y mostrar cada frame con resolución reducida
while cap.isOpened():
    ret, frame = cap.read()  # Leer frame

    if not ret:  # Si no hay más frames, salir del bucle
        print("Fin del video.")
        # 🔹 Liberar recursos
        cap.release()
        cv2.destroyAllWindows()
        break
    
    # Ver el tamaño del frame original
    #print("Tamaño del frame original:", frame.shape)

    # Redimensionar el frame para reducir la resolución
    frame_resized = cv2.resize(frame, (0, 0), fx=scale_factor, fy=scale_factor)
    height, width, _ = frame_resized.shape

    # Recortar el área deseada del frame
    frame_cropped = frame_resized[ymin:ymax, xmin:xmax]

    # Ver el tamaño del frame reducido
    #print("Tamaño del frame reducido:", frame_resized.shape)

    # Convertir a espacio de color HSV
    hsv = cv2.cvtColor(frame_cropped, cv2.COLOR_BGR2HSV)
    
    # Crear máscaras para detectar plastilina amarilla y verde
    mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)
    mask_green = cv2.inRange(hsv, lower_green, upper_green)

    # Copia del frame original para visualizar detecciones
    frame_original = frame_cropped.copy()
    
    # Encontrar contornos de los objetos detectados (amarillo)
    contours_yellow, _ = cv2.findContours(mask_yellow, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours_yellow:
        x, y, w, h = cv2.boundingRect(contour)
        cx, cy = x + w // 2, y + h // 2  # Coordenadas del centro del objeto
        lat, lon = raster_to_geo(cx, cy, width, height, transformer, xmin_geo, ymin_geo, xmax_geo, ymax_geo)
        if lat and lon:
            print(f"Plastilina amarilla detectada en: Raster({cx}, {cy}) -> Geo({lat}, {lon})")
        
        # Dibujar rectángulo alrededor del objeto detectado
        cv2.rectangle(frame_original, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.circle(frame_original, (cx, cy), 5, (0, 0, 255), -1)
    
    # Encontrar contornos de los objetos detectados (verde)
    contours_green, _ = cv2.findContours(mask_green, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours_green:
        x, y, w, h = cv2.boundingRect(contour)
        cx, cy = x + w // 2, y + h // 2  # Coordenadas del centro del objeto
        lat, lon = raster_to_geo(cx, cy, width, height, transformer, xmin_geo, ymin_geo, xmax_geo, ymax_geo)
        if lat and lon:
            print(f"Plastilina verde detectada en: Raster({cx}, {cy}) -> Geo({lat}, {lon})")
        
        # Dibujar rectángulo alrededor del objeto detectado
        cv2.rectangle(frame_original, (x, y), (x + w, y + h), (0, 255, 255), 2)
        cv2.circle(frame_original, (cx, cy), 5, (255, 0, 0), -1)
    
    # Imprimir el número de objetos detectados
    print(f"Plastilina amarilla detectada: {len(contours_yellow)}")
    print(f"Plastilina verde detectada: {len(contours_green)}")
    

    cv2.imshow("Video Original con Detecciones", frame_original)  # Mostrar el video original con detecciones
    cv2.imshow("Video Frame", frame_cropped)  # Mostrar el frame recortado y reducido
    cv2.imshow("Mascara Amarilla", mask_yellow)  # Mostrar la máscara de detección amarilla
    cv2.imshow("Mascara Verde", mask_green)  # Mostrar la máscara de detección verde


    # Presionar 'q' para salir
    if cv2.waitKey(25) & 0xFF == ord('q'):
        print("Video detenido por el usuario.")
        # 🔹 Liberar recursos
        cap.release()
        cv2.destroyAllWindows()
        break